# 62 — Train + eval SID generator (W3)

Fine-tunes Qwen2.5-1.5B-Instruct + LoRA to emit 3 SID tokens for each W2 query.
Smoke mode: 200 steps, ~10 min. Full: 3 epochs over ~90K rows, ~3-4 hr on L4 / ~1-2 hr on Blackwell.

**Prereqs**: W1 + W2 artifacts on Drive at `/content/drive/MyDrive/recsys2026/sid/track_to_sid.parquet`
and `/content/drive/MyDrive/recsys2026/sid_training/{train,val}.parquet`. HF token in Colab Secrets as `HF_TOKEN`.


In [ ]:
# 1) GPU check.
!nvidia-smi | head -20

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

In [ ]:
# 3) HF auth — pull HF_TOKEN from Colab Secrets (same pattern as notebook 61).
# Setup: Colab → 🔑 Secrets pane → add `HF_TOKEN` with notebook access enabled.
import os
from google.colab import userdata
from huggingface_hub import login
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('HF auth OK')

In [ ]:
# 4) Mount Drive + symlink W1/W2 artifacts. Drive subdir naming matches notebook 61:
#   recsys2026_sid_cache              (W1 quantizer output)
#   recsys2026_sid_training_cache     (W2 generator training data)
#   recsys2026_sid_eval_cache         (W3 eval metrics — created here)
#   recsys2026_sid_generator_cache    (W3 LoRA + merged checkpoints)
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=False)

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
SYMLINKS = [
    ('sid',           f'{DRIVE_BASE}/recsys2026_sid_cache'),
    ('sid_training',  f'{DRIVE_BASE}/recsys2026_sid_training_cache'),
    ('sid_eval',      f'{DRIVE_BASE}/recsys2026_sid_eval_cache'),
    ('sid_generator', f'{DRIVE_BASE}/recsys2026_sid_generator_cache'),
]
for local_name, drive_path in SYMLINKS:
    dst = f'{LOCAL_BASE}/{local_name}'
    os.makedirs(drive_path, exist_ok=True)
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(drive_path, dst)
    print(f'symlink: {dst} -> {drive_path}')

# Verify W1 + W2 artifacts visible. If these fail, notebooks 60/61 didn't ship to Drive.
!ls -la experiments/cache/sid/track_to_sid.parquet
!ls -la experiments/cache/sid_training/train.parquet experiments/cache/sid_training/val.parquet

In [ ]:
# 5) Install/upgrade deps. Colab Pro base ships transformers + datasets + torch but
# their preinstalled torchao (0.10.0) is incompatible with recent peft (which requires
# torchao > 0.16). Upgrade torchao explicitly along with peft. Per project memory
# `feedback_colab_library_traps.md` trap #6.
# tensorboard is for the live training-metrics graphs (cell 11 below).
!pip install -q -U \
    "peft>=0.10" \
    "transformers>=4.40" \
    "accelerate>=0.30" \
    "trl>=0.8" \
    "torchao>=0.17" \
    "tensorboard"
import transformers, peft, torch, torchao
print(f'transformers={transformers.__version__}, peft={peft.__version__}, '
      f'torch={torch.__version__}, torchao={torchao.__version__}')

In [ ]:
# 6) Pytest pre-flight on the SID modules. NO output truncation — if anything fails,
# we need to see the full traceback (--tb=long) and ALL collection errors.
!cd /content/recsys2026 && python -m pytest \
    tests/test_sid_vocab.py \
    tests/test_sid_training_format.py \
    tests/test_sid_inference.py \
    tests/test_sid_eval.py \
    -v --tb=long --no-header 2>&1

In [ ]:
# 7) FULL training run — Qwen2.5-1.5B + LoRA + modules_to_save, 3 epochs.
# Wallclock: ~3-4 hr on L4 / ~1-2 hr on Blackwell.
#
# Storage strategy (per discussion):
#   - output_dir on EPHEMERAL Colab disk (NOT Drive) → trainer writes ~975 MB/checkpoint
#     (with --save-best, optimizer state dropped → 5x smaller than default)
#   - --save-best: keep only best+latest checkpoint by val loss (~1.95 GB peak vs ~9.6 GB).
#     Trade-off: NO RESUME (no optimizer state). On L4 (3-4 hr run + Colab disconnects),
#     consider DROPPING --save-best for resumability (default uses ~9.6 GB checkpoints with
#     optimizer state but supports --resume after a crash). On Blackwell (~1-2 hr) the
#     disconnect risk is lower so --save-best is safer.
#   - --merge: push merged 3 GB model to Hub at end
#   - --cleanup-after-push: delete local copies once Hub push succeeds
#   - --results-dir → Drive: persists tensorboard runs/ + results.txt past Colab death (~5 MB)
#
# Net Drive footprint: ~5 MB (logs only). HF Hub footprint: ~4 GB (LoRA + merged).
import os
RESULTS_DIR = '/content/drive/MyDrive/recsys2026_sid_generator_cache/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
LOG_PATH = '/content/drive/MyDrive/recsys2026_sid_generator_cache/training_log_full.txt'

# Blackwell speedup overrides (uncomment if on Blackwell 95GB / A100 80GB):
#   --micro-batch 32 --grad-accum 1 --no-gradient-checkpointing
# Same effective batch (32) → no LR adjustment, but ~3-5x faster wallclock
# (~30-60 min instead of 3-4 hr). On L4 24GB, KEEP defaults below (will OOM otherwise).
!cd /content/recsys2026 && python -u scripts/train_sid_generator.py \
    --output-dir /content/recsys2026_full_run \
    --hub-repo OrRim123/recsys2026-sid-generator-qwen15b-v1 \
    --merge \
    --save-best \
    --cleanup-after-push \
    --results-dir {RESULTS_DIR} \
    --micro-batch 32 \
    --grad-accum 1 \
    --no-gradient-checkpointing \
    2>&1 | tee {LOG_PATH}

In [ ]:
# 8) EVAL — constrained-beam decode over the val parquet (raw slice = matches Blind-A).
# Uses the FULL merged model from cell 7. ~15-25 min for ~760 val rows on L4.
!cd /content/recsys2026 && python -u scripts/eval_sid_generator.py \
    --model-id OrRim123/recsys2026-sid-generator-qwen15b-v1-merged \
    --eval-slice raw \
    2>&1 | tee /content/drive/MyDrive/recsys2026_sid_generator_cache/eval_log_full.txt

In [ ]:
# 9) Read + display final gate metrics.
import json
m = json.load(open('experiments/cache/sid_eval/w3_eval_metrics.json'))
print(json.dumps(m, indent=2))
print()
print('=' * 60)
if m.get('gate_pass'):
    print(f"GATE PASS — mean nDCG@20={m['mean_ndcg_at_20']:.4f} "
          f"(threshold 0.12; delta vs Phase 0 = {m['delta_vs_phase0']:+.4f})")
else:
    print(f"GATE FAIL — mean nDCG@20={m['mean_ndcg_at_20']:.4f} (threshold 0.12)")
    if m.get('paired_bootstrap_ci'):
        ci = m['paired_bootstrap_ci']
        print(f"  paired-bootstrap CI = ({ci['lo']:.4f}, {ci['hi']:.4f})")
print('=' * 60)


## After the run

**Gate pass** (nDCG@20 ≥ 0.12 AND CI lower-bound > 0):
- Merged model is on Hub at `OrRim123/recsys2026-sid-generator-qwen15b-v1-merged`
- Proceed to W4: build `SID_GENERATOR` retrieval class + register `wrrf_bm25_dense_sid_v1`
- Update `MEMORY.md` with the W3 result file

**Gate fail**:
- If point-estimate is close (0.10-0.12) but CI includes 0: more training (5 epochs), check loss curve
- If point-estimate is low (<0.08): likely the W1 SID coarseness biting (3017 unique SIDs limits ceiling).
  Re-run W1 with smaller latent_dim (256→128) + larger codebook (256→512), then re-run W2 + W3.
- If loss diverged: drop LR to 1e-4, re-run.


In [ ]:
# 10) TensorBoard — live training-metrics graphs (loss, grad_norm, lr, eval_loss).
# Single magic that loads BOTH the live in-progress run AND archived past runs:
#   - live: /content/recsys2026_full_run/runs (deleted at end of training by --cleanup-after-push)
#   - archive: /content/drive/MyDrive/recsys2026_sid_generator_cache/results (per-run dirs persist)
# Run this cell ONCE after cell 7 STARTS — the dashboard updates in real time.
# Post-cleanup the live tab will be empty but archive tab keeps every prior run for comparison.
%load_ext tensorboard
%tensorboard --logdir_spec live:/content/recsys2026_full_run/runs,archive:/content/drive/MyDrive/recsys2026_sid_generator_cache/results